# WEPP Debris-Flow Analysis, Trigger Storms to Sediment Response

**PROTECT Task 3 · Risk Index & Resilience Improvement Plan**
**Owner:** Mason Bindl, TRPA Data Team (mbindl@trpa.gov)
**Status:** Working analysis — methodology placeholder pending ICF climate team review
**Companion products:** `tahoe-precip-events.html` (Precipitation Event Explorer), `climate-data.html` (Climate Data Explorer), `PROTECT/climate/` pipeline

## Purpose

Connect observed extreme-precipitation anomalies to hillslope erosion / debris-flow response
using the WEPP model (Water Erosion Prediction Project). Motivating case: the **July 14, 2026
debris flow at (39.12675, −120.24278)** in the Blackwood Canyon area. Analysis chain:

MRMS radar QPE (observed storm) → storm parameterization → WEPP single-event and
continuous runs (unburned vs disturbed slope, current vs climate-intensified) → sediment yield.

## Key prior findings this notebook reproduces

| Finding | Value | Source |
|---|---|---|
| Rain at the failure site, 7/13–15 window | 15.8 mm total, 9.7 mm/hr peak (< 2-yr depth) | MRMS MultiSensor QPE ~1 km |
| WEPP hindcast at site storm: intact slope | 0.0 mm runoff, 0.0 t/ha | this notebook, section 6 |
| WEPP hindcast at site storm: disturbed slope | 5.8 mm runoff, **2.8 t/ha** | this notebook, section 6 |
| Basin ≥10-yr 1-hr burst frequency (obs. 2021–2026) | ~1.4 episodes/yr | MRMS burst catalog |
| Late-century multiplier on ≥10-yr-threshold hours | ×4.6 (SSP3-7.0, WRF 3 km) | Cal-Adapt WRF |

**Interpretation:** the failure required slope susceptibility, not a rare storm. The LS-01/LS-02
pairs should score susceptibility × moderate-storm frequency — the quantity climate change
multiplies fastest.

## 1. Configuration

All run parameters live here (promote to `config.yaml` per TRPA conventions when this
notebook stabilizes). Nothing below this cell should need editing for a standard rerun.

**WEPPPY_DIR** — clone of [github.com/rogerlew/wepppy](https://github.com/rogerlew/wepppy)
(~1.8 GB). We use only two self-contained pieces, so the full WEPPcloud stack (Redis, Docker,
Rust services) is NOT required:

- `wepp_runner/` — builds `.run` files and executes the vendored WEPP binaries
  (`pip install redis` is its one import-time dependency).
- data folders — CLIGEN station parameters, BAER management files, SSURGO-derived soils.

**Binary note:** the repo vendors both Linux (ELF) and Windows (PE32 `.exe`) builds of WEPP
and CLIGEN. On the TRPA Windows workstations set the `.exe` names below; the defaults here
are the Linux names used in the original cloud-session run.

In [ ]:
import sys, re, json, shutil, subprocess
from pathlib import Path
print(sys.executable)  # confirm arcgispro-py3 on TRPA workstations

CONFIG = {
    # --- paths ---
    "wepppy_dir": r"C:\Users\mbindl\Documents\GitHub\wepppy",   # clone of rogerlew/wepppy
    "work_dir":   r"C:\Users\mbindl\Documents\GitHub\PROTECT\wepp\runs_debris_flow",
    # Windows binaries (Linux equivalents: "cligen532", wepp_bin=None -> bin/wepp)
    "cligen_bin": "cligen532.exe",
    "wepp_bin":   "WEPP2014.exe",

    # --- the event and site ---
    "site": {"lat": 39.12675, "lon": -120.24278,
             "name": "2026-07-14 debris flow, Blackwood Canyon area"},
    "event_window_utc": ("2026-07-13T00", "2026-07-16T00"),

    # --- MRMS source (Iowa State mtarchive; MultiSensor Pass 2 archive begins late 2020) ---
    "mrms_url": ("https://mtarchive.geol.iastate.edu/{y}/{m:02d}/{d:02d}/mrms/ncep/"
                 "MultiSensor_QPE_01H_Pass2/MultiSensor_QPE_01H_Pass2_00.00_"
                 "{y}{m:02d}{d:02d}-{h:02d}0000.grib2.gz"),

    # --- climate scaling (from PROTECT/climate WRF 3km analysis, ssp370 2070-2099 vs 1981-2010) ---
    "future_intensity_factor": 1.37,   # +37% on 1-hr extreme rates (2-model mean; PLACEHOLDER pending ICF)

    # --- Atlas 14 anchor thresholds (Meyers 60-min PDS depths, mm) ---
    "t_2yr_1h_mm": 14.3,
    "t_10yr_1h_mm": 22.3,

    # --- CLIGEN ---
    "cligen_station_par": "wepppy/climates/cligen/2015_par_files/ca048758.par",  # "TAHOE CA", 39.17 -120.15, 1898 m
    "cligen_years": 30,

    # --- WEPP scenario matrix ---
    "sim_years_continuous": 30,
    "soil": "wepppy/wepp/soils/soilsdb/data/Database/ca/CAGWIN(LCOS).sol",
    "burned_ke_mmhr": 2.0,        # effective conductivity under high-severity burn (unburned Ke = 14.19)
    "burned_ki_multiplier": 2.0,  # interrill erodibility increase for burned soil
    "man_unburned": "wepppy/wepp/management/data/BAER/Unburned Forest-95_0% Cover.man",
    "man_burned":   "wepppy/wepp/management/data/BAER/High Severity Fire-30_0% Cover.man",
    # slope profile: replace with a DEM-delineated .slp of the actual Blackwood hillslope
    # for production; the test-fixture profile below is a generic steep forest hillslope.
    "slope_file": "wepppy/_tests/feverish-lamp/watershed/hill_23.slp",
}
WPY = Path(CONFIG["wepppy_dir"]); WORK = Path(CONFIG["work_dir"])
(WORK / "runs").mkdir(parents=True, exist_ok=True)
(WORK / "output").mkdir(exist_ok=True)
sys.path.insert(0, str(WPY))

### Why these choices (options you could pick instead)

**Soil — `CAGWIN(LCOS)`.** Loamy coarse sand from decomposed granite: the classic Lake Tahoe
West/South shore soil and the texture most associated with basin debris flows. Alternatives in
the same folder: `TOEM(LCOS)` (shallower granitic, even more erodible — worth a sensitivity
run), or any SSURGO map-unit soil WEPPcloud builds for the real hillslope. The burned variant
edits two numbers in the `.sol` file, following BAER practice: **Ke** (effective hydraulic
conductivity) drops from 14.19 to ~2 mm/hr to represent fire-induced water repellency, and
**Ki** (interrill erodibility) doubles. These two edits are what turn "no runoff ever" into a
realistic post-fire response — they are also the two parameters ICF should sanity-check.

**Management — BAER pair.** `Unburned Forest-95% Cover` vs `High Severity Fire-30% Cover`
(residue/canopy cover drive WEPP's protection of the surface). The repo also has
`Mod Severity Fire-45%` and `Low Severity Fire-80%` for a severity gradient, and a
`data/Tahoe/` folder (Old Forest, thinning-recovery sequences) — note the Tahoe files use a
management format the vendored binaries reject ("tilseq" error); WEPPcloud preprocesses them.
Stick to BAER files for direct binary runs.

**Climate — CLIGEN station `ca048758` "TAHOE CA".** Tahoe City co-op station, 1,898 m,
40 observed years. Nearest alternatives by lat/lon in the 2015 par set: `ca042467` Donner
Memorial Park and `ca040931` Boca (Truckee side). CLIGEN generates a stochastic daily climate
preserving the station's monthly statistics; it does NOT reproduce specific historical years.
For single-event hindcasts we overwrite one day with the observed storm (section 5).

**Slope — fixture vs real.** The `.slp` here is a generic steep profile so the notebook runs
anywhere. For the production hindcast, delineate the actual failure hillslope in WEPPcloud
(it exports `.slp` per hillslope) and drop the file path into CONFIG.

## 2. Environment check and wepp_runner import

`wepp_runner` needs `redis` installed (unused at runtime here, imported at module top).
On the arcgispro-py3 environment: `python -m pip install --user redis`.

In [ ]:
try:
    import redis  # noqa: F401
except ImportError:
    raise SystemExit("pip install --user redis  (wepp_runner imports it)")

from wepp_runner.wepp_runner import make_hillslope_run, run_hillslope
print("wepp_runner OK:", make_hillslope_run.__module__)

# resolve binaries
CLIGEN = WPY / "wepppy/climates/cligen/bin" / CONFIG["cligen_bin"]
assert CLIGEN.exists(), CLIGEN
print("cligen:", CLIGEN)

## 3. Observed storm at the failure site (MRMS)

**Source metadata.** NOAA **MRMS MultiSensor QPE, 1-hour, Pass 2**: radar-derived
precipitation corrected with gauges and model data, ~1 km CONUS grid, hourly GRIB2, archived
by Iowa State's mtarchive. *Pass 2* waits ~1 hr for more gauges (better than Pass 1).
Alternatives: `RadarOnly_QPE_01H` (no gauge correction, longer archive),
`GaugeCorr_QPE_01H` (pre-2020 product name), NCEP Stage IV (hourly, 4 km, back to 2002)
for pre-MRMS events. Radar QPE in complex terrain under-detects shallow orographic precip
and can be beam-blocked — treat cell values as estimates, best for convective storms.

We pull a 5×5-cell neighborhood around the site for the 72-hour event window and reduce to
the center-cell hourly series. Requires `pygrib` (`pip install --user pygrib`).

In [ ]:
import gzip, io, time
import numpy as np, requests, pygrib
import datetime as dt

def fetch_site_series(lat, lon, t0, t1, url_tmpl):
    hours, t = [], dt.datetime.fromisoformat(t0)
    while t <= dt.datetime.fromisoformat(t1):
        hours.append(t); t += dt.timedelta(hours=1)
    WIN = {}
    series, times = [], []
    for ts in hours:
        url = url_tmpl.format(y=ts.year, m=ts.month, d=ts.day, h=ts.hour)
        r = requests.get(url, timeout=90)
        if r.status_code != 200: continue
        p = WORK / "tmp.grib2"; p.write_bytes(gzip.decompress(r.content))
        g = pygrib.open(str(p)); msg = g.message(1)
        vals = np.ma.filled(msg.values, np.nan)
        if not WIN:
            lats, lons = msg.latlons(); lons = (lons + 180) % 360 - 180
            iy, ix = np.unravel_index(np.argmin((lats-lat)**2 + (lons-lon)**2), lats.shape)
            WIN.update(iy=iy, ix=ix)
            print("site cell:", lats[iy, ix], lons[iy, ix])
        v = vals[WIN["iy"], WIN["ix"]]
        series.append(max(float(v), 0.0)); times.append(ts)
        g.close()
    return times, np.array(series)

times, cell = fetch_site_series(CONFIG["site"]["lat"], CONFIG["site"]["lon"],
                                CONFIG["event_window_utc"][0], CONFIG["event_window_utc"][1],
                                CONFIG["mrms_url"])
def roll_max(a, w):
    c = np.convolve(a, np.ones(w), "valid"); return float(c.max()) if len(c) else float(a.sum())
site_storm = {"total_mm": round(float(cell.sum()), 1), "max1h_mm": round(float(cell.max()), 1),
              "max6h_mm": round(roll_max(cell, 6), 1), "max24h_mm": round(roll_max(cell, 24), 1)}
print(site_storm)
print("vs Atlas 14 1-hr depths: 2-yr", CONFIG["t_2yr_1h_mm"], "mm | 10-yr", CONFIG["t_10yr_1h_mm"], "mm")
# Expected (2026-07 session): {'total_mm': 15.8, 'max1h_mm': 9.7, 'max6h_mm': 13.1, 'max24h_mm': 15.8}

## 4. Storm parameterization for WEPP

WEPP's daily climate rows describe each storm with four numbers, which is how we inject an
observed event without breakpoint files:

| Field | Meaning | How we set it |
|---|---|---|
| `prcp` | storm depth (mm) | observed event total at the site |
| `dur` | storm duration (h) | `depth / max1h × 1.5`, clamped 1–24 h (keeps the observed peak-to-mean ratio plausible) |
| `tp` | time-to-peak as fraction of duration | 0.4 (typical convective; sensitivity-test 0.2–0.6) |
| `ip` | peak / average intensity ratio | `max1h / (depth/dur)` — anchors WEPP's internal storm shape to the observed peak hour |

**Higher-fidelity option:** WEPP accepts *breakpoint* climate files (cumulative depth at
arbitrary times), which would carry the actual MRMS hyetograph. WEPPcloud's climate builder
does this from observed records — the production path once the site run moves there.

In [ ]:
def storm_params(depth, max1h):
    dur = min(24.0, max(1.0, depth / max1h * 1.5))
    ip = max(1.01, min(60.0, max1h / (depth / dur)))
    return round(dur, 2), round(ip, 2)

dur, ip = storm_params(site_storm["total_mm"], site_storm["max1h_mm"])
print(f"site storm -> depth {site_storm['total_mm']} mm, dur {dur} h, ip {ip}, tp 0.4")

## 5. Build WEPP inputs

Per WEPP convention, a run directory holds `p{id}.man / .slp / .cli / .sol` plus a sibling
`output/` folder. Three helper builders below:

1. **CLIGEN climate** — 30-year stochastic Tahoe City record (continuous runs and the donor
   file for event runs).
2. **Event climate** — year 1 of the CLIGEN file with every day's precip zeroed except the
   storm day (July 15 slot), which gets the parameterized event. Zeroing other days isolates
   the event (no antecedent moisture — conservative; note in interpretation).
3. **Managements** — the multi-year continuous runs need the BAER 1-year rotation expanded:
   set total years AND rotation repeats AND physically duplicate the Rotation block N times
   (the vendored binaries re-read one block per repeat; without duplication WEPP hits EOF).

In [ ]:
# 5.1 CLIGEN 30-yr Tahoe City climate
par = WPY / CONFIG["cligen_station_par"]
shutil.copy(par, WORK / par.name)
cli30 = WORK / "tahoe30.cli"
subprocess.run([str(CLIGEN), f"-i{par.name}", f"-o{cli30.name}",
                "-b1", f"-y{CONFIG['cligen_years']}", "-t5", "-F"],
               cwd=WORK, check=True, capture_output=True)
print(cli30, cli30.stat().st_size, "bytes")

CLI_LINES = cli30.read_text().splitlines()
HDR, ROWS = CLI_LINES[:15], CLI_LINES[15:]

def event_cli(path, depth, dur, ip, tp=0.4):
    out = list(HDR)
    for l in ROWS:
        p = l.split()
        if len(p) < 12: continue
        if p[2] != "1": break
        if p[0] == "15" and p[1] == "7":
            p[3], p[4], p[5], p[6] = f"{depth:.1f}", f"{dur:.2f}", f"{tp:.2f}", f"{ip:.2f}"
        else:
            p[3] = "0.0"
        out.append(" " + " ".join(f"{x:>6s}" for x in p))
    Path(path).write_text("\n".join(out) + "\n")

def scaled_cli(path, factor, threshold_mm=20.0):
    """Future continuous climate: scale storm days >= threshold by the WRF factor."""
    out = []
    for i, l in enumerate(CLI_LINES):
        p = l.split()
        if i > 14 and len(p) >= 12:
            try:
                prcp = float(p[3])
                if prcp >= threshold_mm:
                    p[3] = f"{prcp*factor:.1f}"
                    l = " " + " ".join(f"{x:>6s}" for x in p)
            except ValueError: pass
        out.append(l)
    Path(path).write_text("\n".join(out) + "\n")

def multi_year_man(src, dst, years):
    t = Path(src).read_text()
    t = t.replace("1 # (total) years in simulation", f"{years} # (total) years in simulation", 1)
    t = t.replace("1  # rotation repeats", f"{years}  # rotation repeats", 1)
    m = re.search(r"(#\n# Rotation 1: year 1 to 1\n#\n)(.*)$", t, re.S)
    body = m.group(2)
    blocks = [f"#\n# Rotation {y}: year {y} to {y}\n#\n" + body for y in range(1, years + 1)]
    Path(dst).write_text(t[:m.start()] + "".join(blocks))

def burned_sol(src, dst, ke, ki_mult):
    raw = Path(src).read_text()
    # CAGWIN line: ... ki=4560328.00 kr=.004893 shcrit=2.36 avke=14.19
    raw = raw.replace("4560328.00", f"{4560328.00*ki_mult:.2f}")
    raw = raw.replace("2.36  14.19", f"2.36  {ke:5.2f}")
    Path(dst).write_text(raw)

print("builders ready")

## 6. Run the matrix

Two experiments, both 2 vegetation × 2 climate:

- **A. Site-storm hindcast** (single event, 1-yr run): the observed 7/14 storm at the failure
  site, and its +37% future twin.
- **B. 30-year continuous** (chronic loading): CLIGEN climate vs future-scaled climate.

Sediment is read from `output/H{id}.loss.dat` — "AVERAGE ANNUAL SEDIMENT LEAVING PROFILE"
in t/ha (for the 1-yr event runs this is the event yield).

In [ ]:
R = WORK / "runs"
def stage(rid, man_path, sol_text, cli_path):
    (R / f"p{rid}.man").write_text(Path(man_path).read_text() if isinstance(man_path, (str, Path)) else man_path)
    (R / f"p{rid}.sol").write_text(sol_text)
    shutil.copy(WPY / CONFIG["slope_file"], R / f"p{rid}.slp")
    shutil.copy(cli_path, R / f"p{rid}.cli")

def parse_loss(rid):
    txt = (WORK / "output" / f"H{rid}.loss.dat").read_text()
    sed = re.search(r"([\d.]+)\s+t/ha \(assuming", txt)
    ro = re.search(r"Mean annual runoff from rainfall\s+([\d.]+)", txt)
    return dict(sed_tha=float(sed.group(1)) if sed else None,
                runoff_mm=float(ro.group(1)) if ro else None)

sol_u = (WPY / CONFIG["soil"]).read_text()
burned_sol(WPY / CONFIG["soil"], WORK / "cagwin_burned.sol",
           CONFIG["burned_ke_mmhr"], CONFIG["burned_ki_multiplier"])
sol_b = (WORK / "cagwin_burned.sol").read_text()

results = []
# --- A. site-storm hindcast (1-yr event runs) ---
rid = 200
for scen, f in [("observed", 1.0), ("future+37%", CONFIG["future_intensity_factor"])]:
    d, m1 = site_storm["total_mm"] * f, site_storm["max1h_mm"] * f
    dur, ip = storm_params(d, m1)
    ecli = WORK / f"event_{scen.replace('%','').replace('+','_')}.cli"
    event_cli(ecli, d, dur, ip)
    for veg, man, sol in [("unburned", WPY / CONFIG["man_unburned"], sol_u),
                          ("burned", WPY / CONFIG["man_burned"], sol_b)]:
        rid += 1
        stage(rid, man, sol, ecli)
        make_hillslope_run(rid, 1, str(R), reveg=False)
        run_hillslope(rid, str(R), wepp_bin=CONFIG["wepp_bin"], timeout=300)
        results.append(dict(experiment="event_hindcast", scen=scen, veg=veg,
                            depth_mm=round(d, 1), max1h_mm=round(m1, 1), **parse_loss(rid)))

# --- B. 30-yr continuous matrix ---
scaled_cli(WORK / "tahoe30_future.cli", 1.25)   # +25% on days >=20mm (daily-scale factor)
Y = CONFIG["sim_years_continuous"]
multi_year_man(WPY / CONFIG["man_unburned"], WORK / "man_u30.man", Y)
multi_year_man(WPY / CONFIG["man_burned"], WORK / "man_b30.man", Y)
rid = 300
for scen, cli in [("current", cli30), ("future+25%", WORK / "tahoe30_future.cli")]:
    for veg, man, sol in [("unburned", WORK / "man_u30.man", sol_u),
                          ("burned", WORK / "man_b30.man", sol_b)]:
        rid += 1
        stage(rid, man, sol, cli)
        make_hillslope_run(rid, Y, str(R), reveg=False)
        run_hillslope(rid, str(R), wepp_bin=CONFIG["wepp_bin"], timeout=600)
        results.append(dict(experiment="continuous_30yr", scen=scen, veg=veg, **parse_loss(rid)))

import pandas as pd
df = pd.DataFrame(results)
df

**Reference results from the 2026-07-31 cloud-session run** (your numbers should match
closely; CLIGEN is seeded deterministically by station file, so continuous runs reproduce):

| experiment | scen | veg | sediment t/ha | runoff mm |
|---|---|---|---|---|
| event_hindcast | observed | unburned | **0.0** | 0.0 |
| event_hindcast | observed | burned | **2.80** | 5.8 |
| event_hindcast | future+37% | unburned | 0.0 | 0.0 |
| event_hindcast | future+37% | burned | **5.05** | 10.7 |
| continuous_30yr | current | unburned | 0.20 | 1.6 |
| continuous_30yr | current | burned | 73.5 | 158 |
| continuous_30yr | future+25% | unburned | 0.77 | 7.4 |
| continuous_30yr | future+25% | burned | 98.5 | 211 |

In [ ]:
# quick chart - TRPA brand colors (validated palette)
import matplotlib.pyplot as plt
ev = df[df.experiment == "event_hindcast"].copy()
labels = ev.veg + "\n" + ev.scen
colors = ["#93BFE7" if v == "unburned" else ("#C45C1A" if "future" in s else "#9C3E27")
          for v, s in zip(ev.veg, ev.scen)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, ev.sed_tha, color=colors)
for x, y in zip(labels, ev.sed_tha):
    ax.text(x, y, f" {y:.2f}", ha="center", va="bottom", fontsize=9, color="#003B71")
ax.set_ylabel("event sediment (t/ha)", color="#003B71")
ax.set_title("July 14 debris-flow site: WEPP single-event hindcast", color="#003B71")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Interpretation

The observed site storm was ordinary — 15.8 mm, 9.7 mm/hr peak, below the 2-yr 1-hr depth,
at a cell with zero cataloged bursts since 2021. WEPP reproduces the asymmetry that explains
the failure anyway: the intact slope infiltrates the entire storm (zero runoff), while the
disturbed slope (fire-type Ke reduction + halved cover) produces runoff and ~2.8 t/ha in a
single event. The +37% future twin nearly doubles the disturbed-slope response and still does
nothing on intact forest.

Consequences for the Task 3 methodology: debris-flow exposure should be modeled as
**slope susceptibility × moderate-storm frequency**, not extreme-storm frequency alone.
Moderate-burst hours are the quantity our WRF analysis says climate multiplies fastest
(×2.9 on 2-yr-threshold hours by 2070–2099, vs ×4.6 on 10-yr but from a smaller base).

### Caveats

Single synthetic hillslope, not the Blackwood profile; BAER-style Ke/Ki edits stand in for
measured burn effects; event runs have no antecedent moisture (conservative); the
parameterized storm shape approximates the MRMS hyetograph; MRMS underestimates are
possible in beam-blocked terrain; the +37%/+25% climate factors are 2-model placeholder
values pending ICF's spec review.

## 8. Production path with WEPPcloud

Defensible site numbers come from **WEPPcloud** ([wepp.cloud](https://wepp.cloud)) on the real
watershed. Settings for the Blackwood hindcast, in UI order:

1. **Configuration:** use the *Lake Tahoe* config if offered (Tahoe-calibrated soils/
   managements and lake sub-basin defaults); otherwise *Disturbed* (BAER-style burn classes).
2. **Extent/delineation:** center on 39.12675, −120.24278; delineate the failure catchment
   (channel critical source area small enough to isolate the initiating hillslope; export the
   hillslope `.slp` back into CONFIG above to upgrade this notebook's profile).
3. **Soils:** SSURGO (auto). Note which map unit covers the scar (expect Cagwin/Toem family).
4. **Landuse/treatments:** run twice — undisturbed baseline, then apply a soil burn severity
   (SBS) map or uniform high-severity treatment on the initiating slope.
5. **Climate:** *observed* daily (gridMET/DAYMET) for context runs; for the event itself use
   single-storm / breakpoint input with the MRMS hyetograph from section 3.
6. **Run WEPP** and export: hillslope loss table, `H*.ebe.dat` events, and the sediment
   delivery GeoJSON — these feed the Risk Index LS-pair exposure layers directly.
7. Record the run URL (WEPPcloud runs are shareable/reproducible by URL) in
   `docs/METHODS.md` and the run registry.

## 9. Provenance

| Item | Value |
|---|---|
| Original analysis session | Cowork, 2026-07-31 |
| MRMS product | MultiSensor_QPE_01H_Pass2, Iowa State mtarchive |
| Atlas 14 | Volume 6 v2, PDS, Meyers point, mm conversions |
| WRF factors | Cal-Adapt cadcat `wrf/ucla` 3 km hourly `prec`, EC-Earth3 + MIROC6, SSP3-7.0 |
| WEPP | vendored binary, wepppy repo (`wepp_runner/bin`) |
| CLIGEN | v5.32300, station ca048758 "TAHOE CA" |
| Companion data | `PROTECT/climate_app/*.json` from session (burst catalog, hindcast, WEPP events) |

Change log: 2026-07-31 initial version (cloud session, Linux binaries); adapt `CONFIG`
binaries for Windows before first workstation run.